# Modular

In [ ]:
!pip install torchmetrics
!pip install torchinfo

In [ ]:
from pathlib import Path
import shutil

shutil.rmtree('modular_folder', ignore_errors=True)
modular_folder = Path('modular_folder')
modular_folder.mkdir(parents=True, exist_ok=True)

## data_setup.py

In [ ]:
%%writefile modular_folder/data_setup.py

import torchvision
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

def create_dataloaders(
    train_dir: str,
    test_dir: str,
    data_transform: torchvision.transforms,
    batch_size: int
) -> tuple[DataLoader, DataLoader, list[str]]:

    # Create dataset
    train_data = ImageFolder(
        root=train_dir,
        transform=data_transform
    )

    test_data = ImageFolder(
        root=test_dir,
        transform=data_transform
    )

    classes = train_data.classes

    # Create dataloader
    train_dataloader = DataLoader(
        dataset = train_data,
        batch_size = batch_size,
        shuffle = True
    )

    test_dataloader = DataLoader(
        dataset = test_data,
        batch_size = batch_size,
        shuffle = True
    )

    ## Verifying data
    image, label = train_data[0]
    print('[INFO] Verifying data...')
    print('Length of the training dataset: ', len(train_data))
    print('Length of the testing dataset: ', len(train_data))
    print('Size of an image: ', image.shape)
    print('Classes: ', classes)

    plt.imshow(image.permute(1,2,0))
    plt.axis('off')
    plt.title(f'{label}: {classes[label]}')
    plt.savefig('test.png')
    
    

    return train_dataloader, test_dataloader, classes



## build_model.py

In [ ]:
%%writefile modular_folder/build_model.py

import torch
import torch.nn as nn
from torchinfo import summary

def build_tinyvgg(
        in_channels: int,
        hidden_units: int,
        out_channels: int,
        in_img_size: int
) -> torch.nn.Module:

    class TinyVGG(nn.Module):
        def __init__(
                self, 
                in_channels: int,
                hidden_units: int,
                out_channels: int,
                in_img_size: int
        ):
            super().__init__()

            self.conv_1 = nn.Sequential(
                nn.Conv2d(
                    in_channels=in_channels,
                    out_channels=hidden_units,
                    kernel_size=3,
                    stride=1,
                    padding=1
                ),
                nn.ReLU(),
                nn.Conv2d(
                    in_channels=hidden_units,
                    out_channels=hidden_units,
                    kernel_size=3,
                    stride=1,
                    padding=1
                ),
                nn.ReLU(),
                nn.MaxPool2d(
                    kernel_size=2
                )
            )

            self.conv_2 = nn.Sequential(
                nn.Conv2d(
                    in_channels=hidden_units,
                    out_channels=hidden_units,
                    kernel_size=3,
                    stride=1,
                    padding=1
                ),
                nn.ReLU(),
                nn.Conv2d(
                    in_channels=hidden_units,
                    out_channels=hidden_units,
                    kernel_size=3,
                    stride=1,
                    padding=1
                ),
                nn.ReLU(),
                nn.MaxPool2d(
                    kernel_size=2
                )
            )


            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(
                    in_features=hidden_units*int(in_img_size/4)*int(in_img_size/4),
                    out_features=1024
                ),
                nn.ReLU(),
                nn.Linear(
                    in_features=1024, out_features=512
                ),
                nn.ReLU(),
                nn.Linear(
                    in_features=512, out_features=256
                ),
                nn.ReLU(),
                nn.Linear(
                    in_features=256, out_features=out_channels
                ),
            )
            
        
        def forward(self, x: torch.Tensor) -> torch.Tensor:
            out1 = self.conv_1(x)
            out2 = self.conv_2(out1)
            out3 = self.classifier(out2)
            return out3

    model = TinyVGG(in_channels, hidden_units, out_channels,in_img_size)

    print(f'\n[INFO]: Verifying model using an input of (1, 3, {in_img_size}, {in_img_size})')
    print(summary(model, input_size=(1, 3, in_img_size, in_img_size)))
    

    return model

## engine.py

In [ ]:
%%writefile modular_folder/engine.py


import torch
import torch.nn as nn
import torchmetrics.classification
from tqdm.auto import tqdm
from typing import Literal
import matplotlib.pyplot as plt


def train_step(
        model: torch.nn.Module,
        data: torch.Tensor,
        label: torch.Tensor,
        loss_fn: torch.nn,
        acc_fn: torchmetrics.classification,
        optimizer: torch.optim,
        num_classes: int
):
    
    model.train()
    y_preds = model(data)
    loss = loss_fn(y_preds, torch.nn.functional.one_hot(label, num_classes=num_classes).to(torch.float32))
    acc = acc_fn(torch.argmax(y_preds, dim=1), label)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item(), acc.item()

def test_step(
        model: torch.nn.Module,
        data: torch.Tensor,
        label: torch.Tensor,
        loss_fn: torch.nn,
        acc_fn: torchmetrics.classification,
        num_classes: int
):
    
    model.eval()
    with torch.inference_mode():
        y_preds = model(data)
        loss = loss_fn(y_preds, torch.nn.functional.one_hot(label, num_classes=num_classes).to(torch.float32))
        acc = acc_fn(torch.argmax(y_preds, dim=1), label)

    return loss.item(), acc.item()


def train(
        epochs: int,
        model: torch.nn.Module,
        train_dataloader: torch.utils.data.DataLoader,
        test_dataloader: torch.utils.data.DataLoader,
        loss_fn: torch.nn,
        acc_fn: torchmetrics.classification,
        optimizer: torch.optim,
        num_classes: int,
        device: Literal['cuda', 'cpu']
):
    plt_epochs = []
    plt_train_loss = []
    plt_train_acc = []
    plt_test_loss = []
    plt_test_acc = []
    

    for epoch in range(epochs):
        
        ep_train_loss = 0
        ep_train_acc = 0
        ep_test_loss = 0
        ep_test_acc = 0

        for (X, y) in tqdm(train_dataloader):
            train_loss, train_acc = train_step(model, X.to(device), y.to(device), loss_fn, acc_fn, optimizer, num_classes)
            ep_train_loss += train_loss
            ep_train_acc += train_acc
        
        for (X, y) in tqdm(test_dataloader):
            test_loss, test_acc = test_step(model, X.to(device), y.to(device), loss_fn, acc_fn, num_classes)
            ep_test_loss += test_loss
            ep_test_acc += test_acc
        
        num_train_batches = len(train_dataloader)
        num_test_batches = len(test_dataloader)
        ep_train_loss /= num_train_batches
        ep_train_acc /= num_train_batches
        ep_test_loss /= num_test_batches
        ep_test_acc /= num_test_batches

        plt_epochs.append(epoch)
        plt_train_loss.append(ep_train_loss)
        plt_train_acc.append(ep_train_acc)
        plt_test_loss.append(ep_test_loss)
        plt_test_acc.append(ep_test_acc)


        print('Epoch: ', epoch)
        print(f'Train Loss: {ep_train_loss:.4f} | Train Accuracy: {ep_train_acc:.2f}')
        print(f'Test Loss: {ep_test_loss:.4f} | Test Accuracy: {ep_test_acc:.2f}')
    
    plt.figure(figsize=(18,5))
    plt.subplot(1,2,1)
    plt.plot(plt_epochs, plt_train_loss, label='Train Loss')
    plt.plot(plt_epochs, plt_test_loss, label='Test Loss')
    plt.title('Losses')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(plt_epochs, plt_train_acc, label='Train Acc')
    plt.plot(plt_epochs, plt_test_acc, label='Test Acc')
    plt.title('Losses')
    plt.legend()
    plt.savefig('plot.png')

        


## Utils

In [ ]:
%%writefile modular_folder/utils.py

import torch
from . import build_model
import torchvision
from PIL import Image
from typing import Literal
import matplotlib.pyplot as plt
from pathlib import Path


def save_model(
        model: torch.nn.Module,
        save_name: str
):  
    print(f'\n[INFO] Saving to {save_name}')
    if not (save_name.endswith('.pt') or save_name.endswith('.pth')):
        raise ValueError("Provided save_name does not end with neither .pt nor .pth.")
    
    save_folder = Path('model')
    save_folder.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), save_folder / save_name)


def load_model(
        model_name: str,
        in_channels: int,
        hidden_units: int,
        out_channels: int,
        in_img_size: int
) -> torch.nn.Module:
    if not (model_name.endswith('.pt') or model_name.endswith('.pth')):
        raise ValueError("Provided model_name does not end with neither .pt nor .pth.")

    save_folder = Path('model')
    save_folder.mkdir(parents=True, exist_ok=True)
    model = build_model.build_tinyvgg(in_channels, hidden_units, out_channels, in_img_size)
    model.load_state_dict(torch.load((save_folder / model_name), weights_only=True))
    return model


def infer(
        model: torch.nn.Module,
        img_path: str,
        transform: torchvision.transforms,
        device: Literal['cuda', 'cpu'],
        classes: list[str]
):
    
    img_tensor = transform(Image.open(Path(img_path))).unsqueeze(dim=0).to(device)
    img_class = Path(img_path).parent.stem
    

    model = model.to(device)

    model.eval()
    with torch.inference_mode():
        y_preds = model(img_tensor)
        y_probs = torch.softmax(y_preds, dim=1).squeeze()
        y_label = torch.argmax(y_probs)
        y_class = classes[y_label]
    
    plt.figure(figsize=(9,5))
    plt.imshow(img_tensor.squeeze().permute(1,2,0).cpu())
    plt.title(f'GT: {img_class} | Pred: {y_class} | Probs: {y_probs[y_label]:.2f}')
    plt.savefig('infer.png')


    


## Script

In [ ]:
%%writefile modular_folder/train.py


import torch
from . import data_setup, build_model, engine, utils
from torchvision import transforms
from torchmetrics.classification import MulticlassAccuracy
from pathlib import Path
import glob
import random
import argparse

# Hyperparameter - replaced by argparse

parser = argparse.ArgumentParser()
parser.add_argument('--img_size', type=int)
parser.add_argument('--batch_size', type=int)
parser.add_argument('--epochs', type=int)
parser.add_argument('--lr', type=float)
parser.add_argument('--in_channels', default=3, type=int)
parser.add_argument('--hidden_units', default=32, type=int)
parser.add_argument('--model_name', type=str)

args = parser.parse_args()
IMG_SIZE = args.img_size
BATCH_SIZE = args.batch_size
EPOCHS = args.epochs
LEARNING_RATE = args.lr
IN_CHANNELS = args.in_channels
HIDDEN_UNITS = args.hidden_units
MODEL_NAME = args.model_name


device = 'cuda' if torch.cuda.is_available() else 'cpu'
data_folder_name = 'data'
data_folder = Path(data_folder_name)
data_folder.mkdir(parents=True, exist_ok=True)


# Create dataloader
simple_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

train_dataloader, test_dataloader, classes = data_setup.create_dataloaders(
    train_dir = data_folder / 'train',
    test_dir = data_folder / 'test',
    data_transform = simple_transform,
    batch_size = BATCH_SIZE
)


# Build a model
model_0 = build_model.build_tinyvgg(
    in_channels = IN_CHANNELS,
    hidden_units= HIDDEN_UNITS,
    out_channels=len(classes),
    in_img_size=IMG_SIZE
).to(device)

# Define utils
optimizer = torch.optim.Adam(
    model_0.parameters(), LEARNING_RATE
)

loss_fn = torch.nn.CrossEntropyLoss()

acc_fn = MulticlassAccuracy(
    num_classes=len(classes), average='micro'
).to(device)

# Training / Testing
engine.train(
    epochs = EPOCHS,
    model= model_0,
    train_dataloader= train_dataloader,
    test_dataloader= test_dataloader,
    loss_fn= loss_fn,
    acc_fn= acc_fn,
    optimizer= optimizer,
    num_classes= len(classes),
    device= device
)

# Save model
utils.save_model(model_0, MODEL_NAME)

# Load model
loaded_model = utils.load_model(
        model_name=MODEL_NAME,
        in_channels=IN_CHANNELS,
        hidden_units=HIDDEN_UNITS,
        out_channels=len(classes),
        in_img_size=IMG_SIZE
).to(device)

# Infer
all_test_img_paths = glob.glob(str(data_folder / 'test/*/*.jpg'))

utils.infer(
        model = loaded_model,
        img_path = random.choice(all_test_img_paths),
        transform = simple_transform,
        device = device,
        classes = classes
)




## Setup data

In [ ]:
# Get Data

from pathlib import Path
import requests
import zipfile
import os

data_folder = Path('data')
data_folder.mkdir(parents=True, exist_ok=True)
zip_file = data_folder / 'pizza_sushi_steak.zip'

if not zip_file.is_file():
    request = requests.get('https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip')
    with open(zip_file, 'wb') as f:
        f.write(request.content)
    
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(data_folder)

print('data')
print(os.listdir(data_folder))
print('\nTrain')
print(os.listdir(data_folder / 'train'))
print(os.listdir(data_folder / 'train' / 'steak'))
print('\nTest')
print(os.listdir(data_folder / 'test'))
print(os.listdir(data_folder / 'test' / 'steak'))

## Run script

In [ ]:
!python -m modular_folder.train --img_size 256 --batch_size 32 --epochs 100 --lr 1e-5 --hidden_units 32 --model_name 'model_0.pt'

In [ ]:
## Showcase a result since it is now run as a script
from IPython.display import Image
Image('test.png')


In [ ]:
Image('plot.png')

In [ ]:
Image('infer.png')